# Memento Team 端到端系统测试

测试完整编排链路：
- 加载 `.env`
- 初始化 `OrchestratorAgent`（LangChain + MCP）
- 编排器分解任务 → Worker 并行执行（每个 Worker 是一个 Memento-S agent）
- 输出最终结果

In [1]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

# 定位仓库根目录
ROOT = Path.cwd().resolve()
if not (ROOT / "main.py").exists():
    ROOT = Path("/Users/steven/memento-team")

sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "Memento-S"))

load_dotenv(ROOT / ".env")

from orchestrator.orchestrator_agent import OrchestratorAgent
from core.workboard import cleanup_board

query = """
	帮我实现一个多智能体博弈的代码库
"""


async def run_system_test(user_query: str) -> str:
    # 清理历史 workboard
    try:
        cleanup_board()
    except Exception:
        pass

    model = ChatOpenAI(
        model=os.getenv("OPENROUTER_MODEL", "google/gemini-3-flash-preview"),
        openai_api_key=os.getenv("OPENROUTER_API_KEY"),
        openai_api_base=os.getenv("OPENROUTER_BASE_URL"),
        temperature=0,
    )

    orchestrator = OrchestratorAgent(
        model=model,
        command=sys.executable,
        args=[str(ROOT / "orchestrator" / "mcp_server.py")],
    )
    await orchestrator.start()
    try:
        result = await orchestrator.run(user_query)
        return result["output"]
    finally:
        await orchestrator.close()


output = await run_system_test(query)
print(output)

我已为您实现了一个模块化的多智能体博弈（Multi-Agent Game）代码库。该库采用了解耦的设计，方便您后续扩展不同的环境逻辑或强化学习算法。

### 1. 代码库结构
代码保存在 `multi_agent_game/` 目录下：
- `environment.py`: 核心游戏环境。实现了一个基础的网格世界（Grid-world），包含状态管理、步进逻辑（step）和观测生成。
- `agent.py`: 智能体基类及实现。包含 `BaseAgent` 抽象类、`RandomAgent`（随机决策）以及 `LearningAgent`（学习型智能体占位符）。
- `engine.py`: 交互引擎。负责协调环境与多个智能体之间的循环，处理动作分发和奖励收集。
- `main.py`: 入口文件。演示了如何初始化环境、配置智能体并运行多个回合（Episodes）。

### 2. 核心设计说明
- **环境接口**: 遵循类似 OpenAI Gym 的 `reset` 和 `step` 模式，但支持多智能体字典输入 `{agent_id: action}` 和输出。
- **灵活性**: `GameEngine` 可以接受任何实现了 `act` 方法的智能体字典，支持异构智能体（例如：一个随机智能体 vs 一个 AI 智能体）。
- **可扩展性**: 您可以通过继承 `BaseAgent` 并实现 `act` 方法来快速集成 DQN、PPO 等强化学习算法。

### 3. 运行与验证
经过自动化测试验证，代码库可以正常运行。
**运行命令**:
```bash
cd multi_agent_game
python3 main.py
```

**预期输出示例**:
```text
Simulation Results:
Episode 1: Total Rewards = {0: 20.0, 1: 20.0}
Episode 2: Total Rewards = {0: 20.0, 1: 20.0}
...
```

### 4. 后续扩展建议
- **复杂环境**: 在 `environment.py` 中修改 `step` 函数，加入智能体碰撞、资源竞争或零和博弈（Zero-sum）的奖励逻辑。
- **训练逻辑**: 在 `LearningAgen